In [1]:
import logging
import os

import requests
from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.executors import SequentialToolExecutor
from tavily import TavilyClient

load_dotenv()

# Strands logs a warning on every turn because gpt-oss returns reasoning text. It's harmless.
logging.getLogger("strands.models.openai").setLevel(logging.ERROR)

# Groq's API is OpenAI-compatible, so Strands' OpenAIModel works with Groq's base_url.
groq_model = OpenAIModel(
    model_id="openai/gpt-oss-120b",
    client_args={
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1"
    }
)

In [2]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def get_weather(city: str) -> dict:
    """Gets the current weather for a city from OpenWeather.

    Args:
        city: The city name (e.g., 'Kathmandu').
    """
    print(f"\n[WEATHER TOOL] {city}")
    try:
        response = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={"q": city, "appid": os.getenv("OPENWEATHER_API_KEY"), "units": "metric"},
            timeout=10,
        )
    except requests.RequestException as e:
        # Don't return str(e). It contains the request URL, and the URL has the API key.
        return {"error": f"Could not reach OpenWeather ({type(e).__name__})."}

    if response.status_code != 200:
        return {"error": f"OpenWeather returned {response.status_code} for '{city}'."}

    data = response.json()
    return {
        "city": data["name"],
        "temperature_c": data["main"]["temp"],
        "description": data["weather"][0]["description"],
    }


@tool
def search_attractions(city: str) -> dict:
    """Searches the web for popular tourist attractions and their entry fees in a city.

    Args:
        city: The city name (e.g., 'Kathmandu').
    """
    print(f"\n[SEARCH TOOL] attractions in {city}")
    try:
        response = tavily_client.search(
            query=f"top 3 tourist attractions in {city} with entry ticket price",
            search_depth="basic",
            max_results=3,
        )
    except Exception as e:
        return {"error": f"Search failed: {e}"}

    return {
        "results": [
            {"title": r.get("title"), "content": r.get("content"), "url": r.get("url")}
            for r in response.get("results", [])
        ]
    }


@tool
def calculate_total_cost(costs: list[float]) -> dict:
    """Adds up a list of costs (e.g., entry fees in USD) and returns the total.

    Args:
        costs: The individual costs to add, e.g. [10.0, 5.5, 0].
    """
    total = round(sum(costs), 2)
    print(f"\n[CALCULATOR TOOL] sum({costs}) = {total}")
    return {"costs": costs, "total": total}

In [3]:
TRAVEL_PLANNER_PROMPT = """You are a travel planner that builds one-day itineraries.
For the city the user gives you:
1. Call get_weather for the current weather.
2. Call search_attractions and pick 3 popular attractions. Estimate each one's entry fee
   in USD from the search results (use 0 if it is free; say when a price is an estimate).
3. Call calculate_total_cost with the 3 fees. Never add the numbers yourself.
4. Reply with a concise one-day itinerary:
   - Weather summary and a tip based on it
   - Morning / Afternoon / Evening, one attraction each, with its fee
   - Total estimated cost (from the calculator tool)
If a tool returns an error, say so briefly and continue with what you have."""

planner_agent = Agent(
    model=groq_model,
    tools=[get_weather, search_attractions, calculate_total_cost],
    tool_executor=SequentialToolExecutor(),
    system_prompt=TRAVEL_PLANNER_PROMPT,
    callback_handler=None,  # don't stream the model's reasoning; print only the tool calls and final plan
)

response = planner_agent("Plan a one-day trip to Kathmandu.")

print("\n--- One-Day Itinerary ---")
print(response)


[WEATHER TOOL] Kathmandu



[SEARCH TOOL] attractions in Kathmandu



[CALCULATOR TOOL] sum([8.0, 1.5, 3.5]) = 13.0



--- One-Day Itinerary ---
**Kathmandu – One‑Day Itinerary**

**Weather:** Overcast clouds, around 19 °C.  
*Tip:* Cool and partly cloudy – great for walking, just bring a light jacket and an umbrella just in case of a drizzle.

---

### Morning  
**Kathmandu Durbar Square** – Entry fee US $8 (foreigners).  
Explore the historic palace complex, temples, and bustling market squares while the city wakes up.

### Afternoon  
**Swayambhunath (Monkey Temple)** – Entry fee US $1.5.  
Climb the stupa for panoramic views of the valley; watch the playful monkeys and enjoy the serene atmosphere.

### Evening  
**Boudhanath Stupa** – Entry fee US $3.5.  
Visit the world’s largest stupa at sunset; circle the prayer hall, spin the prayer wheels, and soak in the tranquil vibe.

---

**Total estimated cost:** US $13.0 (calculated).

